# Simple Crossover Backtest (Updated)

Este notebook demonstra como usar `Account` e `Operation` para rodar um backtest simples de
crossover de médias móveis, usando o ambiente interno de backtest.

- Ambiente de backtest centralizado em `Account` e `Operation`.
- Atualização de candles via `Operation.backtest_update_candles()` com decorator que dispara eventos.
- Abertura/fechamento de posições via métodos de `Operation`.
- Estado da conta lido diretamente em `backtest_account_data` (`MqlAccountInfo`).


In [1]:
from datetime import datetime, timezone
import pandas as pd
import numpy as np

from algo_trading.sources.MetaTrader5_source.account.account import Account
from algo_trading.sources.MetaTrader5_source.models.metatrader import (
    MqlAccountInfo,
    ENUM_ACCOUNT_TRADE_MODE, ENUM_ACCOUNT_STOPOUT_MODE, ENUM_ACCOUNT_MARGIN_MODE,
    ENUM_SYMBOL_CALC_MODE, ENUM_SYMBOL_SWAP_MODE, ENUM_POSITION_TYPE,
)


## Inicialização do ambiente de backtest
`login_backtest()` requer `live_account_data` preenchido. Simulamos um live mínimo (sem MT5 real).


In [2]:
# Substitua com suas credenciais do MT5
LOGIN = 5041151393             # int
SERVER = "MetaQuotes-Demo" # str (ex.: "MetaQuotes-Demo")
PASSWORD = "V*Fo2qVq"  # str
MT5_PATH = ""                 # opcional: caminho do terminal

account = Account()
# Isto inicializa mt5.initialize(...) internamente
_live = account.login_live(login=LOGIN, server=SERVER, password=PASSWORD, path=MT5_PATH)
_live

# Cria a conta de backtest
backtest = account.login_backtest(balance=10000, leverage=100)
operation = backtest.operation
backtest.simulated_spread = 0  # opcional: ajustar spread simulado


2025-10-11 16:51:58,701 - INFO - Successfully logged in to live account #5041151393
2025-10-11 16:51:58,711 - INFO - Live account data successfully updated.
2025-10-11 16:51:58,715 - INFO - Backtest account successfully created.


## Cadastro de símbolos no ambiente
Antes de atualizar candles, cadastre os símbolos.


In [3]:
symbols = ["EURUSD"]
operation.backtest_add_symbol_data(symbols)
operation.backtest_symbols_data.head()


,tick_size,contract_size,trade_calc_mode,swap_mode,swap_long,swap_short,volume_min,volume_max,volume_step,volume_limit,swap_rollover3days,last_candle
symbol,,,,,,,,,,,,
EURUSD,0.00001,100000.0,0,1,-0.7,-1.0,0.01,500.0,0.01,0.0,3,None


## Dados e indicadores (SMA crossover)
Gera dados sintéticos OHLC e calcula SMAs.


In [ ]:
np.random.seed(0)
n = 300
base = 1.1000
noise = np.cumsum(np.random.normal(0, 0.0005, n))
close = base + noise
high = close + 0.0008
low = close - 0.0008
open_ = np.roll(close, 1)
open_[0] = close[0]
index = pd.date_range("2025-01-01", periods=n, freq="T", tz=timezone.utc)

df = pd.DataFrame({"open": open_, "high": high, "low": low, "close": close}, index=index)

fast = 20
slow = 50
df["sma_fast"] = df["close"].rolling(fast).mean()
df["sma_slow"] = df["close"].rolling(slow).mean()

symbol = "EURUSD"

def current_position_type():
    if not backtest.positions:
        return None
    return backtest.positions[0].type

def flat():
    return len(backtest.positions) == 0

df.head()


## Loop de backtest: atualização de candles e sinais
O decorator em `backtest_update_candles` aciona os eventos do ambiente automaticamente.


In [ ]:
for ts, row in df.iterrows():
    last_candle = pd.Series({
        "open": row["open"], 
        "high": row["high"], 
        "low": row["low"], 
        "close": row["close"]
    }, name=ts)

    # Atualiza o ambiente (dispara eventos)
    operation.backtest_update_candles({symbol: last_candle})

    sma_f = row["sma_fast"]
    sma_s = row["sma_slow"]
    if pd.isna(sma_f) or pd.isna(sma_s):
        continue

    pos = current_position_type()
    bullish = sma_f > sma_s
    bearish = sma_f < sma_s

    if bullish and flat():
        operation.buy(symbol=symbol, volume=1.0, comment=f"SMA{fast}>{slow}")
    elif bearish and not flat() and pos.name == ENUM_POSITION_TYPE.POSITION_TYPE_BUY.name:
        operation.close_all_positions(comment="Cross down")
    elif bearish and flat():
        operation.sell(symbol=symbol, volume=1.0, comment=f"SMA{fast}<{slow}")
    elif bullish and not flat() and pos.name == ENUM_POSITION_TYPE.POSITION_TYPE_SELL.name:
        operation.close_all_positions(comment="Cross up")


## Estado final da conta e posições


In [ ]:
print("Saldo:", backtest.balance)
print("Equity:", backtest.equity)
print("Posições abertas:", len(backtest.positions))
print("Ordens pendentes:", len(backtest.orders))

if backtest.positions:
    p = backtest.positions[0]
    print("Primeira posição:", p.symbol, p.type.name, p.volume, p.price_open)

operation.backtest_symbols_data.tail()
